# D&D Beyond Background Sync

Sync homebrew backgrounds from Google Sheets to D&D Beyond.

## Setup Instructions

1. Get your D&D Beyond cookies (Copy as cURL and extract Cookie header).
2. Get security tokens from the create background page.
3. Update your `.env` with `DDB_COOKIES`, `DDB_SECURITY_TOKEN`, and `DDB_AUTHENTICITY_TOKEN`.


## 1. Setup and Imports

In [ ]:
import pandas as pd
import requests
import json
import time
import re
import os
from typing import Dict, List, Optional
from dotenv import load_dotenv
from datetime import datetime
from pathlib import Path
import sys

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Spreadsheet.sheets import ContentSheetsClient

from DNDBeyond.core.Helpers.DnDBeyondAPI import DnDBeyondAPI
from DNDBeyond.core.Helpers import convert_background_to_ddb, normalize_ddb_id

print("✓ Imports loaded")


## 2. D&D Beyond Authentication

In [ ]:
# ============================================
# D&D Beyond Configuration
# ============================================

load_dotenv()

DDB_BASE_URL = os.getenv("DDB_BASE_URL") or "https://www.dndbeyond.com"
DDB_COOKIES = os.getenv("DDB_COOKIES")
DDB_SECURITY_TOKEN = os.getenv("DDB_SECURITY_TOKEN")
DDB_AUTHENTICITY_TOKEN = os.getenv("DDB_AUTHENTICITY_TOKEN")
REQUEST_VERIFICATION_TOKEN = os.getenv("REQUEST_VERIFICATION_TOKEN")
DDB_USER_ID = os.getenv("DDB_USER_ID")
DDB_USERNAME = os.getenv("DDB_USERNAME")

# ============================================
# Setup Session
# ============================================

session = requests.Session()

if DDB_COOKIES:
    session.headers.update({
        "Cookie": DDB_COOKIES,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:147.0) Gecko/20100101 Firefox/147.0",
        "Referer": f"{DDB_BASE_URL}/homebrew/creations/create-background/create",
        "Origin": DDB_BASE_URL,
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-User": "?1",
        "Upgrade-Insecure-Requests": "1"
    })

    print("✓ Using cookie authentication")
    print(f"✓ Base URL: {DDB_BASE_URL}")
    print(f"✓ User: {DDB_USERNAME} (ID: {DDB_USER_ID})")

    if DDB_SECURITY_TOKEN and DDB_AUTHENTICITY_TOKEN:
        print("✓ Security tokens configured")
    else:
        print("⚠️  Security tokens not set - you'll need these to create backgrounds!")
else:
    print("✗ ERROR: No cookies provided!")
    print("   Please set DDB_COOKIES with your browser session cookies")


## 3. Load Background Data

In [ ]:
print("Loading backgrounds from Google Sheets...")
client = ContentSheetsClient("non-fantasy", credentials_path=str(repo_root / "FiveETools" / "key.json"))
df_backgrounds = client.get_sheet_by_name("backgrounds")

background_list = []
for _, row in df_backgrounds.iterrows():
    name = row.get("Background")
    if not name or pd.isna(name):
        continue
    items = []
    skills = row.get("Skills")
    if pd.notna(skills) and str(skills).strip():
        items.append({"type": "item", "name": "Skill Proficiencies", "entry": str(skills)})
    tools = row.get("Tools")
    if pd.notna(tools) and str(tools).strip():
        items.append({"type": "item", "name": "Tool Proficiencies", "entry": str(tools)})
    languages = row.get("Languages")
    if pd.notna(languages) and str(languages).strip():
        items.append({"type": "item", "name": "Languages", "entry": str(languages)})
    equipment = row.get("Items")
    if pd.notna(equipment) and str(equipment).strip():
        items.append({"type": "item", "name": "Equipment", "entry": str(equipment)})

    entries = []
    if items:
        entries.append({"type": "list", "style": "list-hang-notitle", "items": items})

    feature_name = row.get("Feature Name")
    feature = row.get("Feature")
    if pd.notna(feature_name) and str(feature_name).strip():
        entries.append({"type": "entries", "name": str(feature_name), "entries": [str(feature) if pd.notna(feature) else ""], "data": {"isFeature": True}})

    background_list.append({
        "name": str(name),
        "entries": entries,
    })

print(f"✓ Loaded {len(background_list)} backgrounds")
if background_list:
    print(f"Sample background: {background_list[0].get('name')}")


## 4. D&D Beyond API Helper

In [ ]:
ddb_api = DnDBeyondAPI(
    session,
    DDB_SECURITY_TOKEN,
    DDB_AUTHENTICITY_TOKEN,
    verification_token=REQUEST_VERIFICATION_TOKEN,
)
print("✓ D&D Beyond API initialized")


## 5. Sync Backgrounds to D&D Beyond

In [ ]:
DRY_RUN = False
BATCH_SIZE = 1
DELAY = 1
VERBOSE = True
SKIP_EXISTING = True
UPDATE_EXISTING = True

print("=" * 60)
if DRY_RUN:
    print("DRY RUN - No backgrounds will be created/updated")
else:
    print("⚠️  LIVE MODE - Backgrounds WILL be created/updated!")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Batch size: {'ALL BACKGROUNDS' if BATCH_SIZE is None else BATCH_SIZE}")
print(f"Delay between requests: {DELAY}s")
print(f"Verbose logging: {'ON' if VERBOSE else 'OFF'}")
print(f"Skip existing: {'YES' if SKIP_EXISTING else 'NO (will attempt to create anyway)'}")
print(f"Update existing: {'YES (will update when DDB ID exists)' if UPDATE_EXISTING else 'NO'}")
print()

backgrounds_to_sync = background_list if BATCH_SIZE is None else background_list[:BATCH_SIZE]
print(f"Total backgrounds to process: {len(backgrounds_to_sync)}")
print()

print("Checking spreadsheet for existing DDB IDs...")
BACKGROUND_NAME_COLUMN = "Background"
background_to_ddb_id = {}
if 'DDB' in df_backgrounds.columns and BACKGROUND_NAME_COLUMN in df_backgrounds.columns:
    for _, row in df_backgrounds.iterrows():
        background_name = row.get(BACKGROUND_NAME_COLUMN)
        ddb_id = row.get('DDB')
        if background_name and pd.notna(ddb_id):
            normalized_id = normalize_ddb_id(ddb_id)
            if normalized_id:
                background_to_ddb_id[background_name] = normalized_id
    print(f"✓ Found {len(background_to_ddb_id)} backgrounds with DDB IDs in spreadsheet")
else:
    print("  No DDB IDs found in spreadsheet")

print("\nFetching existing homebrew backgrounds from D&D Beyond...")
if not DRY_RUN:
    try:
        existing_backgrounds = ddb_api.backgrounds.list()
        print(f"✓ Found {len(existing_backgrounds)} existing homebrew backgrounds\n")
    except Exception as exc:
        print(f"⚠️  Could not fetch existing backgrounds: {exc}")
        existing_backgrounds = []
else:
    existing_backgrounds = []

REQUIRED_FORM_FIELDS = set()
if not DRY_RUN:
    try:
        response = session.get(f"{DDB_BASE_URL}/homebrew/creations/create-background/create", timeout=30)
        REQUIRED_FORM_FIELDS = set(re.findall(r'name="([^"]+)"[^>]*data-validation-required="true"', response.text))
        print("✓ Refreshed required background fields from live page")
    except Exception as exc:
        print(f"⚠️  Could not refresh required background fields: {exc}")

results = {
    'created': 0,
    'updated': 0,
    'skipped': 0,
    'errors': 0,
    'details': []
}

spreadsheet_updates = []

for i, background in enumerate(backgrounds_to_sync, 1):
    background_name = background.get('name', 'Unnamed')
    print(f"[{i}/{len(backgrounds_to_sync)}] Processing: {background_name}")

    try:
        if VERBOSE:
            print("  → Converting background to D&D Beyond format...")
        try:
            ddb_form = convert_background_to_ddb(background)
            if VERBOSE:
                print(f"    Short description length: {len(ddb_form.get('short-description-wysiwyg', ''))}")
        except Exception as conv_error:
            print(f"  ✗ Conversion failed: {conv_error}")
            results['errors'] += 1
            results['details'].append({
                'title': background_name,
                'action': 'error',
                'success': False,
                'error': f"Conversion error: {str(conv_error)}",
                'error_type': 'conversion'
            })
            continue

        missing_fields = [
            key for key in REQUIRED_FORM_FIELDS
            if key not in ddb_form or str(ddb_form.get(key, '')) == ''
        ]
        if missing_fields:
            print("  ✗ Missing required fields for create")
            print(f"    Missing fields: {missing_fields}")
            results['errors'] += 1
            results['details'].append({
                'title': background_name,
                'action': 'error',
                'success': False,
                'error_type': 'validation_failed',
                'missing_fields': missing_fields
            })
            continue

        existing_id = background_to_ddb_id.get(background_name)
        if existing_id:
            if UPDATE_EXISTING and not SKIP_EXISTING:
                if DRY_RUN:
                    print(f"  → DRY RUN: Would update background (ID: {existing_id})")
                    results['updated'] += 1
                    results['details'].append({
                        'title': background_name,
                        'action': 'dry_run_update',
                        'success': True,
                        'id': existing_id
                    })
                else:
                    print(f"  → Updating existing background (ID: {existing_id})")
                    slug = DnDBeyondAPI.create_slug(background_name)
                    updated = ddb_api.backgrounds.update(existing_id, slug, {'form_data': ddb_form})
                    if updated:
                        print("  ✓ Updated background successfully")
                        results['updated'] += 1
                        results['details'].append({
                            'title': background_name,
                            'action': 'updated',
                            'success': True,
                            'id': existing_id
                        })
                    else:
                        print("  ✗ Failed to update background")
                        results['errors'] += 1
                        results['details'].append({
                            'title': background_name,
                            'action': 'error',
                            'success': False,
                            'error_type': 'update_failed',
                            'id': existing_id,
                            'error': ddb_api.last_error
                        })
                if i < len(backgrounds_to_sync):
                    time.sleep(DELAY)
            else:
                print(f"  ✓ Already synced (DDB ID: {existing_id})")
                results['skipped'] += 1
                results['details'].append({
                    'title': background_name,
                    'action': 'skipped',
                    'success': True,
                    'id': existing_id,
                    'reason': 'already_in_spreadsheet'
                })
            continue

        if SKIP_EXISTING and not DRY_RUN and existing_backgrounds:
            existing_id = ddb_api.backgrounds.find_by_name(background_name, existing_backgrounds)
            if existing_id:
                print(f"  ⚠️  Background exists on D&D Beyond (ID: {existing_id})")
                spreadsheet_updates.append({
                    'match_value': background_name,
                    'update_column': 'DDB',
                    'update_value': existing_id
                })
                results['skipped'] += 1
                results['details'].append({
                    'title': background_name,
                    'action': 'skipped',
                    'success': True,
                    'id': existing_id,
                    'reason': 'already_exists_ddb'
                })
                continue

        if DRY_RUN:
            print("  → DRY RUN: Would create background")
            results['created'] += 1
            results['details'].append({
                'title': background_name,
                'action': 'dry_run_create',
                'success': True
            })
        else:
            print("  → Creating background on D&D Beyond...")
            background_id = ddb_api.backgrounds.create({'form_data': ddb_form})
            if background_id:
                print(f"  ✓ Created: ID {background_id}")
                spreadsheet_updates.append({
                    'match_value': background_name,
                    'update_column': 'DDB',
                    'update_value': background_id
                })
                results['created'] += 1
                results['details'].append({
                    'title': background_name,
                    'action': 'created',
                    'success': True,
                    'id': background_id
                })
                existing_backgrounds.append({'name': background_name, 'id': background_id})
            else:
                print("  ✗ Failed to create background")
                results['errors'] += 1
                results['details'].append({
                    'title': background_name,
                    'action': 'error',
                    'success': False,
                    'error_type': 'creation_failed',
                    'error': ddb_api.last_error
                })
            if i < len(backgrounds_to_sync):
                time.sleep(DELAY)

    except Exception as exc:
        print(f"  ✗ Unexpected error: {exc}")
        results['errors'] += 1
        results['details'].append({
            'title': background_name,
            'action': 'error',
            'success': False,
            'error_type': 'unexpected_exception',
            'error': str(exc)
        })

# Write DDB IDs back to spreadsheet
if spreadsheet_updates and not DRY_RUN:
    print('\n' + '=' * 60)
    print('Updating Google Sheets with DDB IDs...')
    print('=' * 60)
    try:
        update_results = client.batch_update_cells_by_row_match(
            ContentSheetsClient.SHEET_GIDS['non-fantasy']['backgrounds'],
            BACKGROUND_NAME_COLUMN,
            spreadsheet_updates
        )
        success_count = sum(1 for v in update_results.values() if v)
        print(f"✓ Updated {success_count}/{len(spreadsheet_updates)} backgrounds in spreadsheet")
    except Exception as exc:
        print(f"✗ Error updating spreadsheet: {exc}")

print()
print('=' * 60)
print('SYNC COMPLETE')
print('=' * 60)
if DRY_RUN:
    print(f"DRY RUN: Would create {results['created']} backgrounds")
    print(f"DRY RUN: Would update {results['updated']} backgrounds")
else:
    print(f"✓ Created: {results['created']}")
    print(f"✓ Updated: {results['updated']}")
    print(f"⚠️  Skipped (already exist): {results['skipped']}")
    print(f"✗ Errors: {results['errors']}")

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_file = f"out/background_sync_log_{timestamp}.json"
log_dir = os.path.dirname(log_file)
os.makedirs(log_dir, exist_ok=True)
with open(log_file, 'w') as f:
    json.dump({
        'timestamp': timestamp,
        'dry_run': DRY_RUN,
        'verbose': VERBOSE,
        'batch_size': BATCH_SIZE,
        'skip_existing': SKIP_EXISTING,
        'update_existing': UPDATE_EXISTING,
        'total': len(backgrounds_to_sync),
        'created': results['created'],
        'updated': results['updated'],
        'skipped': results['skipped'],
        'errors': results['errors'],
        'details': results['details']
    }, f, indent=2)

print(f"\n✓ Log saved to: {log_file}")
